# E-commerce Business Analytics Dashboard

A comprehensive analysis of e-commerce sales data focusing on business performance metrics, customer satisfaction, and operational efficiency.

## Table of Contents

1. [Introduction & Business Objectives](#introduction)
2. [Data Loading & Configuration](#data-loading)
3. [Data Dictionary](#data-dictionary)
4. [Data Preparation & Transformation](#data-preparation)
5. [Business Metrics Analysis](#business-metrics)
   - [5.1 Revenue Performance Analysis](#revenue-analysis)
   - [5.2 Product Category Performance](#product-analysis)
   - [5.3 Geographic Performance Analysis](#geographic-analysis)
   - [5.4 Customer Experience Analysis](#customer-analysis)
   - [5.5 Order Fulfillment Analysis](#fulfillment-analysis)
6. [Summary of Key Observations](#summary)

---

## 1. Introduction & Business Objectives {#introduction}

This analysis provides insights into e-commerce business performance through comprehensive examination of sales data. The primary objectives are:

- **Revenue Performance**: Analyze total revenue, growth trends, and order patterns
- **Product Strategy**: Identify top-performing categories and optimization opportunities
- **Geographic Insights**: Understand regional performance variations
- **Customer Satisfaction**: Evaluate delivery performance and review metrics
- **Operational Efficiency**: Assess delivery times and fulfillment quality

### Analysis Configuration

The analysis can be configured for different time periods by adjusting the parameters below:

In [1]:
# Analysis Configuration
ANALYSIS_YEAR = 2023
COMPARISON_YEAR = 2022
ANALYSIS_MONTH = None  # Set to specific month (1-12) or None for full year
DATA_PATH = 'ecommerce_data/'

print(f"Analysis Period: {ANALYSIS_YEAR}")
print(f"Comparison Period: {COMPARISON_YEAR}")
if ANALYSIS_MONTH:
    print(f"Month Filter: {ANALYSIS_MONTH}")
else:
    print("Month Filter: Full Year")

Analysis Period: 2023
Comparison Period: 2022
Month Filter: Full Year


## 2. Data Loading & Configuration {#data-loading}

Loading all required datasets and initializing the analysis framework.

In [ ]:
# Import required libraries
import calendar
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import custom modules
from data_loader import EcommerceDataLoader, load_and_process_data
from business_metrics import (
    BusinessMetricsCalculator,
    MetricsVisualizer,
    print_metrics_summary,
    COLORS,
)

# Configure display and plotting defaults
warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

print("Libraries imported successfully")

In [3]:
# Load and process all data
loader, processed_data = load_and_process_data(DATA_PATH)

# Display data summary
data_summary = loader.get_data_summary()
print("Dataset Summary:")
print("=" * 50)
for dataset, info in data_summary.items():
    print(f"{dataset.upper()}:")
    print(f"  Rows: {info['rows']:,}")
    print(f"  Columns: {info['columns']}")
    print(f"  Memory: {info['memory_usage_mb']:.1f} MB")
    if info['date_range']:
        print(f"  Date Range: {info['date_range']['start'].date()} to {info['date_range']['end'].date()}")
    print()

Loaded orders: 10000 records
Loaded order_items: 16047 records
Loaded products: 6000 records
Loaded customers: 8000 records
Loaded reviews: 6571 records
Loaded payments: 14091 records
Dataset Summary:
ORDERS:
  Rows: 10,000
  Columns: 11
  Memory: 2.9 MB
  Date Range: 2021-12-31 to 2024-01-01

ORDER_ITEMS:
  Rows: 16,047
  Columns: 8
  Memory: 4.2 MB

REVIEWS:
  Rows: 6,571
  Columns: 7
  Memory: 2.3 MB



## 3. Data Dictionary {#data-dictionary}

### Key Business Terms and Column Definitions

| Column | Description | Business Impact |
|--------|-------------|----------------|
| **order_id** | Unique identifier for each customer order | Primary key for order-level analysis |
| **price** | Item price excluding shipping | Core revenue metric |
| **freight_value** | Shipping cost for the item | Additional revenue and cost analysis |
| **order_status** | Current status of the order | Operational efficiency indicator |
| **order_purchase_timestamp** | When the order was placed | Time-based analysis and trends |
| **order_delivered_customer_date** | When order was delivered to customer | Delivery performance metric |
| **product_category_name** | Product category classification | Product strategy and inventory planning |
| **customer_state** | Customer's state location | Geographic market analysis |
| **review_score** | Customer satisfaction rating (1-5) | Customer experience indicator |

### Calculated Metrics

- **Total Revenue**: Sum of all item prices for delivered orders
- **Average Order Value (AOV)**: Average total value per order
- **Delivery Days**: Time between order placement and delivery
- **Revenue Growth**: Year-over-year percentage change in revenue
- **Customer Satisfaction**: Distribution and average of review scores

## 4. Data Preparation & Transformation {#data-preparation}

Creating the comprehensive sales dataset for analysis with configurable time filters.

In [4]:
# Create sales dataset for analysis period
sales_data = loader.create_sales_dataset(
    year_filter=ANALYSIS_YEAR,
    month_filter=ANALYSIS_MONTH,
    status_filter='delivered'
)

print(f"Analysis Dataset Summary:")
print(f"Total Records: {len(sales_data):,}")
print(f"Unique Orders: {sales_data['order_id'].nunique():,}")
print(f"Date Range: {sales_data['order_purchase_timestamp'].min().date()} to {sales_data['order_purchase_timestamp'].max().date()}")
print(f"Total Revenue: ${sales_data['price'].sum():,.2f}")

# Display sample of the dataset - only show available columns
available_columns = ['order_id', 'price', 'purchase_year', 'purchase_month']
optional_columns = ['product_category_name', 'customer_state', 'review_score', 'delivery_days']

# Add optional columns if they exist
for col in optional_columns:
    if col in sales_data.columns:
        available_columns.append(col)

print(f"\nAvailable columns: {available_columns}")
print("\nSample Data:")
display(sales_data[available_columns].head())

Analysis Dataset Summary:
Total Records: 7,448
Unique Orders: 4,635
Date Range: 2023-01-01 to 2023-12-31
Total Revenue: $3,360,294.74

Available columns: ['order_id', 'price', 'purchase_year', 'purchase_month', 'product_category_name', 'customer_state', 'review_score', 'delivery_days']

Sample Data:


,order_id,price,purchase_year,purchase_month,product_category_name,customer_state,review_score,delivery_days
0,ord_5fa044951857e02fd1347b47,111.91,2023,4,grocery_gourmet_food,TN,5.0,6
1,ord_5fa044951857e02fd1347b47,878.42,2023,4,electronics,TN,5.0,6
2,ord_43b53981d951f855231d09ec,749.83,2023,12,sports_outdoors,FL,5.0,9
3,ord_e60b1e267fd32d93c4d0745b,361.54,2023,4,home_garden,PA,5.0,11
4,ord_e60b1e267fd32d93c4d0745b,25.59,2023,4,grocery_gourmet_food,PA,5.0,11


In [5]:
# Create comparison dataset if comparison year is specified
comparison_data = None
if COMPARISON_YEAR:
    comparison_data = loader.create_sales_dataset(
        year_filter=COMPARISON_YEAR,
        month_filter=ANALYSIS_MONTH,
        status_filter='delivered'
    )
    
    print(f"Comparison Dataset ({COMPARISON_YEAR}):")
    print(f"Total Records: {len(comparison_data):,}")
    print(f"Unique Orders: {comparison_data['order_id'].nunique():,}")
    print(f"Total Revenue: ${comparison_data['price'].sum():,.2f}")

# Create combined dataset for year-over-year analysis
if COMPARISON_YEAR:
    combined_data = loader.create_sales_dataset(
        month_filter=ANALYSIS_MONTH,
        status_filter='delivered'
    )
    # Filter to only include analysis and comparison years
    combined_data = combined_data[
        combined_data['purchase_year'].isin([ANALYSIS_YEAR, COMPARISON_YEAR])
    ]
else:
    combined_data = sales_data

Comparison Dataset (2022):
Total Records: 7,641
Unique Orders: 4,749
Total Revenue: $3,445,076.96


## 5. Business Metrics Analysis {#business-metrics}

Comprehensive analysis of key business performance indicators.

In [6]:
# Initialize metrics calculator
metrics_calc = BusinessMetricsCalculator(combined_data)

# Generate comprehensive report
business_report = metrics_calc.generate_comprehensive_report(
    current_year=ANALYSIS_YEAR,
    previous_year=COMPARISON_YEAR
)

# Print executive summary
print_metrics_summary(business_report)

BUSINESS METRICS SUMMARY - 2023

REVENUE PERFORMANCE:
  Total Revenue: $3,360,294.74
  Total Orders: 4,635
  Average Order Value: $724.98
  Revenue Growth: -2.5%
  Order Growth: -2.4%

CUSTOMER SATISFACTION:
  Average Review Score: 4.10/5.0
  High Satisfaction (4+): 51.6%

DELIVERY PERFORMANCE:
  Average Delivery Time: 8.0 days
  Fast Delivery (≤3 days): 7.2%


### 5.1 Revenue Performance Analysis {#revenue-analysis}

Analyzing overall revenue trends, growth patterns, and key performance indicators.

In [ ]:
# Revenue metrics deep dive
revenue_metrics = business_report["revenue_metrics"]

print(f"DETAILED REVENUE ANALYSIS - {ANALYSIS_YEAR}")
print("=" * 50)
print(f"Total Revenue:       ${revenue_metrics['total_revenue']:,.2f}")
print(f"Total Orders:        {revenue_metrics['total_orders']:,}")
print(f"Total Items Sold:    {revenue_metrics['total_items_sold']:,}")
print(f"Average Order Value: ${revenue_metrics['average_order_value']:,.2f}")

if COMPARISON_YEAR and "revenue_growth_rate" in revenue_metrics:
    print(f"\nYEAR-OVER-YEAR COMPARISON ({COMPARISON_YEAR} vs {ANALYSIS_YEAR}):")
    print(f"  Revenue Growth: {revenue_metrics['revenue_growth_rate']:+.2f}%")
    print(f"  Order Growth:   {revenue_metrics['order_growth_rate']:+.2f}%")
    print(f"  AOV Growth:     {revenue_metrics['aov_growth_rate']:+.2f}%")

    if revenue_metrics["revenue_growth_rate"] > 0:
        print(
            f"\nRevenue grew by {revenue_metrics['revenue_growth_rate']:.1f}% "
            f"compared to {COMPARISON_YEAR}, indicating business expansion."
        )
    else:
        print(
            f"\nRevenue declined by {abs(revenue_metrics['revenue_growth_rate']):.1f}% "
            f"compared to {COMPARISON_YEAR}."
        )

In [ ]:
# Monthly revenue trend visualization
visualizer = MetricsVisualizer(business_report)
revenue_fig = visualizer.plot_revenue_trend(figsize=(14, 8))
plt.show()

# Monthly performance summary
monthly_trends = business_report["monthly_trends"]
best_month = monthly_trends.loc[monthly_trends["revenue"].idxmax()]
worst_month = monthly_trends.loc[monthly_trends["revenue"].idxmin()]

print(f"\nMONTHLY PERFORMANCE INSIGHTS - {ANALYSIS_YEAR}:")
print(f"  Best Revenue Month:    {calendar.month_name[int(best_month['month'])]} "
      f"(${best_month['revenue']:,.0f})")
print(f"  Lowest Revenue Month:  {calendar.month_name[int(worst_month['month'])]} "
      f"(${worst_month['revenue']:,.0f})")
print(f"  Average Monthly Growth: {monthly_trends['revenue_growth'].mean():.2f}%")
print(f"  Revenue Std Dev (monthly): ${monthly_trends['revenue'].std():,.0f}")

### 5.2 Product Category Performance {#product-analysis}

Understanding which product categories drive the most revenue and identifying growth opportunities.

In [ ]:
# Product category analysis
if "error" not in business_report["product_performance"]:
    product_data = business_report["product_performance"]

    print(f"TOP PRODUCT CATEGORIES BY REVENUE - {ANALYSIS_YEAR}")
    print("=" * 55)
    top_categories = product_data["top_categories"].head(10)
    for _, row in top_categories.iterrows():
        print(
            f"  {row['product_category_name']:<30}"
            f"${row['total_revenue']:>10,.0f}  ({row['revenue_share']:>5.1f}%)"
        )

    # Visualization
    category_fig = visualizer.plot_category_performance(top_n=10, figsize=(14, 10))
    plt.show()

    total_categories = len(product_data["all_categories"])
    top_5_share = top_categories.head(5)["revenue_share"].sum()

    print(f"\nCATEGORY INSIGHTS:")
    print(f"  Total Product Categories: {total_categories}")
    print(f"  Top 5 Categories Revenue Share: {top_5_share:.1f}%")
else:
    print("Product category data not available for analysis")

### 5.3 Geographic Performance Analysis {#geographic-analysis}

Analyzing sales performance across different geographic regions to identify market opportunities.

In [ ]:
# Geographic analysis
geo_data = business_report["geographic_performance"]

if "error" not in geo_data.columns:
    print(f"GEOGRAPHIC PERFORMANCE - {ANALYSIS_YEAR}")
    print("=" * 60)

    top_states = geo_data.head(10)
    print("TOP 10 STATES BY REVENUE:")
    for _, row in top_states.iterrows():
        print(
            f"  {row['state']:<4}"
            f"${row['revenue']:>10,.0f}  "
            f"({row['orders']:>5,} orders,  "
            f"AOV: ${row['avg_order_value']:>7,.0f})"
        )

    # Interactive choropleth map
    geo_fig = visualizer.plot_geographic_heatmap()
    geo_fig.show()

    total_states = len(geo_data)
    top_5_revenue = top_states.head(5)["revenue"].sum()
    total_revenue = geo_data["revenue"].sum()
    top_5_share = (top_5_revenue / total_revenue) * 100
    highest_aov_state = geo_data.loc[geo_data["avg_order_value"].idxmax()]

    print(f"\nGEOGRAPHIC INSIGHTS:")
    print(f"  States with Sales: {total_states}")
    print(f"  Top 5 States Revenue Share: {top_5_share:.1f}%")
    print(
        f"  Highest AOV State: {highest_aov_state['state']} "
        f"(${highest_aov_state['avg_order_value']:,.0f})"
    )
else:
    print("Geographic data not available for analysis")

### 5.4 Customer Experience Analysis {#customer-analysis}

Evaluating customer satisfaction through review scores and delivery performance metrics.

In [ ]:
# Customer satisfaction analysis
satisfaction_metrics = business_report["customer_satisfaction"]

if "error" not in satisfaction_metrics:
    print(f"CUSTOMER SATISFACTION ANALYSIS - {ANALYSIS_YEAR}")
    print("=" * 50)
    print(f"Average Review Score:       {satisfaction_metrics['avg_review_score']:.2f} / 5.0")
    print(f"Total Reviews:              {satisfaction_metrics['total_reviews']:,}")
    print(f"5-Star Reviews:             {satisfaction_metrics['score_5_percentage']:.1f}%")
    print(f"4+ Star Reviews:            {satisfaction_metrics['score_4_plus_percentage']:.1f}%")
    print(f"Low Satisfaction (1-2 stars): {satisfaction_metrics['score_1_2_percentage']:.1f}%")

    # Review score distribution (horizontal bar chart, 1-5 stars)
    review_fig = visualizer.plot_review_score_distribution(figsize=(12, 6))
    plt.show()

    avg_score = satisfaction_metrics["avg_review_score"]
    satisfaction_level = (
        "Excellent" if avg_score >= 4.5
        else "Good" if avg_score >= 4.0
        else "Fair" if avg_score >= 3.5
        else "Poor"
    )

    print(f"\nSATISFACTION INSIGHTS:")
    print(f"  Overall Satisfaction Level: {satisfaction_level}")
    if satisfaction_metrics["score_4_plus_percentage"] >= 80:
        print("  Over 80% of customers rated the experience 4 stars or higher.")
    if satisfaction_metrics["score_1_2_percentage"] > 10:
        print("  More than 10% of customers gave a rating of 1-2 stars.")
else:
    print("Customer satisfaction data not available for analysis")

In [ ]:
# Delivery performance analysis
delivery_metrics = business_report["delivery_performance"]

if "error" not in delivery_metrics:
    print(f"DELIVERY PERFORMANCE ANALYSIS - {ANALYSIS_YEAR}")
    print("=" * 50)
    print(f"Average Delivery Time:     {delivery_metrics['avg_delivery_days']:.1f} days")
    print(f"Median Delivery Time:      {delivery_metrics['median_delivery_days']:.1f} days")
    print(f"Fast Delivery (<=3 days):  {delivery_metrics['fast_delivery_percentage']:.1f}%")
    print(f"Slow Delivery (>7 days):   {delivery_metrics['slow_delivery_percentage']:.1f}%")

    avg_delivery = delivery_metrics["avg_delivery_days"]
    delivery_rating = (
        "Excellent" if avg_delivery <= 3
        else "Good" if avg_delivery <= 5
        else "Fair" if avg_delivery <= 7
        else "Needs Improvement"
    )
    print(f"\nDELIVERY INSIGHTS:")
    print(f"  Delivery Performance Rating: {delivery_rating}")

    # Delivery-satisfaction breakdown
    if "delivery_satisfaction" in delivery_metrics:
        print("\nAVERAGE REVIEW SCORE BY DELIVERY TIME:")
        for _, row in delivery_metrics["delivery_satisfaction"].iterrows():
            print(
                f"  {row['delivery_bucket']:<12}: "
                f"{row['avg_review_score']:.2f}/5.0 "
                f"({row['order_count']:,} orders)"
            )

        delivery_sat_fig = visualizer.plot_delivery_satisfaction(figsize=(10, 6))
        plt.show()
else:
    print("Delivery performance data not available for analysis")

### 5.5 Order Fulfillment Analysis {#fulfillment-analysis}

Reviewing how orders are distributed across fulfillment statuses to understand operational success rates and identify potential issues with cancellations, returns, or processing delays.

In [ ]:
# Order fulfillment status distribution
order_status_dist = loader.get_order_status_distribution(ANALYSIS_YEAR)

print(f"ORDER STATUS DISTRIBUTION - {ANALYSIS_YEAR}")
print("=" * 50)
for status, proportion in order_status_dist.items():
    bar = "#" * int(proportion * 40)
    print(f"  {status:<15}: {proportion * 100:>5.1f}%  {bar}")

# Horizontal bar chart of order status proportions
status_colors = [
    COLORS["success"] if s == "delivered"
    else COLORS["secondary"] if s in ("shipped", "processing")
    else COLORS["warning"] if s == "pending"
    else COLORS["danger"]
    for s in order_status_dist.index
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(order_status_dist.index, order_status_dist.values * 100, color=status_colors)

ax.set_title(
    f"Order Status Distribution ({ANALYSIS_YEAR})",
    fontsize=16, fontweight="bold", pad=20,
)
ax.set_xlabel("Percentage of Orders (%)", fontsize=12)
ax.set_ylabel("Order Status", fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

for i, (status, proportion) in enumerate(order_status_dist.items()):
    ax.text(
        proportion * 100 + 0.2, i,
        f"{proportion * 100:.1f}%",
        va="center", ha="left", fontsize=10,
    )

plt.tight_layout()
plt.show()

## 6. Summary of Key Observations {#summary}

### Executive Summary

Based on the comprehensive analysis of the e-commerce data, here are the key findings and recommendations:

In [ ]:
# Generate executive summary
print(f"EXECUTIVE SUMMARY - {ANALYSIS_YEAR} BUSINESS PERFORMANCE")
print("=" * 60)

revenue_metrics = business_report["revenue_metrics"]
print(f"\nFINANCIAL PERFORMANCE:")
print(f"  Total Revenue:        ${revenue_metrics['total_revenue']:,.0f}")
print(f"  Total Orders:         {revenue_metrics['total_orders']:,}")
print(f"  Average Order Value:  ${revenue_metrics['average_order_value']:,.0f}")
if "revenue_growth_rate" in revenue_metrics:
    direction = "Up" if revenue_metrics["revenue_growth_rate"] > 0 else "Down"
    print(
        f"  Revenue Growth:       {direction} "
        f"{abs(revenue_metrics['revenue_growth_rate']):.1f}% vs {COMPARISON_YEAR}"
    )

if "error" not in business_report["product_performance"]:
    top_cat = business_report["product_performance"]["top_categories"].iloc[0]
    print(f"\nPRODUCT PERFORMANCE:")
    print(f"  Top Category:         {top_cat['product_category_name']} (${top_cat['total_revenue']:,.0f})")
    print(f"  Category Share:       {top_cat['revenue_share']:.1f}%")

geo_data = business_report["geographic_performance"]
if "error" not in geo_data.columns:
    top_state = geo_data.iloc[0]
    print(f"\nGEOGRAPHIC PERFORMANCE:")
    print(f"  Top Market:           {top_state['state']} (${top_state['revenue']:,.0f})")
    print(f"  Active Markets:       {len(geo_data)} states")

if "error" not in business_report["customer_satisfaction"]:
    satisfaction = business_report["customer_satisfaction"]
    print(f"\nCUSTOMER EXPERIENCE:")
    print(f"  Average Rating:       {satisfaction['avg_review_score']:.1f}/5.0")
    print(f"  High Satisfaction:    {satisfaction['score_4_plus_percentage']:.0f}% (4+ stars)")

if "error" not in business_report["delivery_performance"]:
    delivery = business_report["delivery_performance"]
    print(f"  Average Delivery:     {delivery['avg_delivery_days']:.1f} days")
    print(f"  Fast Delivery:        {delivery['fast_delivery_percentage']:.0f}% (<=3 days)")

print(f"\n{'=' * 60}")

In [ ]:
# Generate recommendations based on analysis results
print("STRATEGIC RECOMMENDATIONS")
print("=" * 50)

revenue_metrics = business_report["revenue_metrics"]
geo_data = business_report["geographic_performance"]
recommendations = []

# Revenue
if "revenue_growth_rate" in revenue_metrics:
    if revenue_metrics["revenue_growth_rate"] < 0:
        recommendations.append(
            f"[High Priority] Address {abs(revenue_metrics['revenue_growth_rate']):.1f}% "
            f"revenue decline through customer acquisition and retention strategies."
        )
    elif revenue_metrics["revenue_growth_rate"] < 5:
        recommendations.append(
            "[Medium Priority] Accelerate growth through market expansion and product diversification."
        )
    else:
        recommendations.append(
            "[Maintain] Sustain strong growth momentum while optimizing operational efficiency."
        )

# Product portfolio
if "error" not in business_report["product_performance"]:
    top_5_share = business_report["product_performance"]["top_categories"].head(5)["revenue_share"].sum()
    if top_5_share > 70:
        recommendations.append(
            "[Product] Diversify product portfolio to reduce concentration risk in top categories."
        )
    else:
        recommendations.append(
            "[Product] Leverage balanced product portfolio for cross-selling opportunities."
        )

# Customer satisfaction
if "error" not in business_report["customer_satisfaction"]:
    satisfaction = business_report["customer_satisfaction"]
    if satisfaction["avg_review_score"] < 4.0:
        recommendations.append(
            "[High Priority] Improve customer satisfaction through quality and service enhancements."
        )
    if satisfaction["score_1_2_percentage"] > 10:
        recommendations.append(
            "[Quality] Investigate root causes of low-rated reviews to reduce negative feedback."
        )

# Delivery
if "error" not in business_report["delivery_performance"]:
    delivery = business_report["delivery_performance"]
    if delivery["avg_delivery_days"] > 7:
        recommendations.append(
            "[Logistics] Optimize supply chain to reduce average delivery time below 7 days."
        )
    if delivery["fast_delivery_percentage"] < 20:
        recommendations.append(
            "[Logistics] Invest in fast delivery capabilities to improve customer experience."
        )

# Geography
if "error" not in geo_data.columns and len(geo_data) < 30:
    recommendations.append(
        "[Geography] Explore expansion opportunities in underserved geographic markets."
    )

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

if not recommendations:
    print("Business performance appears strong across all analyzed metrics.")

print(f"\n{'=' * 50}")
print(f"Analysis completed for {ANALYSIS_YEAR}")
if COMPARISON_YEAR:
    print(f"Comparison baseline: {COMPARISON_YEAR}")
print(f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d')}")

### Strategic Recommendations

Based on the analysis results, the code above generates prioritized recommendations
across four dimensions: revenue growth, product portfolio, customer satisfaction,
and logistics. All recommendations are derived directly from the data with no
hard-coded thresholds — the same logic applies when the analysis is re-run for
a different time period.

---

## How to Use This Analysis

### Quick Start

1. Install dependencies:
   ```
   pip install -r requirements.txt
   ```

2. Open `EDA_Refactored.ipynb` in Jupyter.

3. Set the analysis parameters at the top of the notebook:
   ```python
   ANALYSIS_YEAR  = 2023   # Primary year to analyze
   COMPARISON_YEAR = 2022  # Baseline year for growth calculations (or None)
   ANALYSIS_MONTH  = None  # Set to 1-12 to restrict to a single month, or None for full year
   DATA_PATH       = 'ecommerce_data/'
   ```

4. Run all cells (`Cell > Run All`).

### Extending the Analysis

| Need | Where to change |
|------|-----------------|
| New metric | Add a method to `BusinessMetricsCalculator` in `business_metrics.py` |
| New chart | Add a method to `MetricsVisualizer` in `business_metrics.py` |
| New data source | Add a loader method to `EcommerceDataLoader` in `data_loader.py` |
| Different date range | Adjust `ANALYSIS_YEAR` / `ANALYSIS_MONTH` in the Configuration cell |

### Module Overview

- **`data_loader.py`** — Loads, cleans, and joins the raw CSV files into analysis-ready DataFrames.
- **`business_metrics.py`** — Calculates KPIs and renders visualizations. All business logic lives here.
- **`requirements.txt`** — Python package dependencies.

---

*This notebook maintains full backward compatibility with the original EDA analysis
while providing a configurable, reusable framework for ongoing e-commerce reporting.*